# General

All project work in IND320 will result in personal hand-ins and online apps.

### 1. Jupyter Notebook

A Jupyter Notebook run locally on your computer, later with access to online and local databases.

This will be your basic development and documentation platform.

#### Required content

- A brief description of AI usage.
- A 300–500-word log describing the compulsory work, including both Jupyter Notebook and Streamlit experience.
- Links to your public GitHub repository and Streamlit app for the compulsory work.

#### Documentation and submission

- Document headings should be clear and usable for navigation during development.
- All code blocks must include enough comments to be understandable and reproducible if someone inherits your project.
- All code blocks must be run before exporting to PDF so that messages and plots are shown.
- Add the `.ipynb` file to the GitHub repository containing your Streamlit project.

### 2. Streamlit App

A Streamlit app running at `https://[yourproject].streamlit.app/`.

- This will be an online version of the project, accessing data exported to CSV format and, later, an online database.
- The code must be hosted on GitHub.
- The code must include relevant comments from the Jupyter Notebook and additional comments regarding Streamlit usage.

#### Project Structure and Collaboration

- There are four parts in the project work, building on each other and resulting in a final portfolio and app to be presented at the end of the semester.
- Cooperation is applauded, and the use of AI tools is encouraged.

## Tasks

### GitHub and Streamlit.app accounts

- GitHub: https://github.com/danielytsan/IND320APP
- Steamlit: https://ind320app-s2bytfwvqxbzmwkxrm2aze.streamlit.app

# Presentation link

 Due to the video file being to large for github I have choosen to upload it to youtube. Use this link to access it:

https://youtu.be/zhCRaJ-9PeA


## AI usage
AI has been used during development of the content provided for this app and Jupyter Notebook

The model I have been using is ChatGPT-6.1 Sol to:
- debugging and suggestion for code with error messages
- learning streamlit and different function
- learing to use api keys and how to format the files for it. Specially for how to use the token on Entso-e.
- help with using spark


# Jupyter Notebook
* Navigate to https://atlas.nve.no to locate and download the shapefiles corresponding to the electricity areas and precipitation areas that correspond to the Magasinstatistikk data. Note: East is split into RS+RØ, and North is split into RM+RN.
    * Randomly sample 1000 locations in each region to use for weather data download.

* Use the open-meteo API to retrieve hourly historical reanalysis data using the ERA5 model for a single location as follows:
    * Create a function for the API download task that takes a region identifier plus a time interval as input, looks up the corresponding list of pairs of longitude and latitude values and downloads at least precipitation and temperature, but include any relevant property you find.

    * Apply the function to download data for NO3 for 2024.

* Outliers and anomalies:
    * Plot the average precipitation as a function of time.
        * Perform a high-pass filtering of the precipitation using the Discrete Cosine Transform to create seasonally adjusted temperature variations (SATV).
        
        * Add curves to the plot indicating Statistical Process Control boundaries between inliers and outliers based on the SATV according to robust statistics estimated from the whole year. Colour outliers with a contrasting colour. Do not plot SATV values; only use them to find boundaries and outliers.
        
        * Let the frequency cut-off for the DCT and the number of standard deviations be parameters with sensible defaults.
        
        * Wrap this in a function that returns the plot and relevant summaries of the outliers, and test the function.

    * Plot the average temperature as a function of time.
        * Indicate anomalies according to the Local Outlier Factor method.
        
        * Let the proportion of outliers be a parameter defaulting to 1%.

        * Wrap this in a function that returns the plot and relevant summaries of the outliers, and test the function.

* Seasonal-Trend decomposition using LOESS (STL)
    * Perform STL on the degree of filling from the magasinstatistikk data and plot its decomposition.
    
    * Let the seasonal smoother, trend smoother and robust (true/false) be parameters, and give each of them sensible defaults.
    
    * Wrap this in a function that returns the plot, and test the function.

* Spectrogram
    * Create a spectrogram based on the same data.
    
    * Let the window length and window overlap be parameters, and give each of them sensible defaults.
    
    * Wrap this in a function that returns the plot, and test the function.

In [1]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"hourly": "temperature_2m",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 52.512603759765625°N 13.419516563415527°E
Elevation: 38.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  temperature_2m
0   2026-11-04 00:00:00+00:00          6.2955
1   2026-11-04 01:00:00+00:00          6.4955
2   2026-11-04 02:00:00+00:00          6.6955
3   2026-11-04 03:00:00+00:00          6.5955
4   2026-11-04 04:00:00+00:00          6.2455
..                        ...             ...
163 2026-11-10 19:00:00+00:00          5.6455
164 2026-11-10 20:00:00+00:00          5.3955
165 2026-11-10 21:00:00+00:00          5.1955
166 2026-11-10 22:00:00+00:00          5.0455
167 2026-11-10 23:00:00+00:00          4.9455

[168 rows x 2 columns]
